In [34]:
import mlflow
from mlflow.tracking import MlflowClient
from mlflow.entities import ViewType
from sklearn.metrics import mean_squared_error
import pandas as pd

In [41]:
MLFLOW_TRACKING_URI = 'http://localhost:5000'

client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

In [6]:
client.search_experiments()

[<Experiment: artifact_location='mlflow-artifacts:/3', creation_time=1774809293004, experiment_id='3', last_update_time=1774809293004, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, workspace='default'>,
 <Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1774791552191, experiment_id='2', last_update_time=1774791552191, lifecycle_stage='active', name='diabetes', tags={}, workspace='default'>,
 <Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1774790643460, experiment_id='1', last_update_time=1774790643460, lifecycle_stage='active', name='nyc-taxi', tags={}, workspace='default'>,
 <Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1774790134789, experiment_id='0', last_update_time=1774790134789, lifecycle_stage='active', name='Default', tags={}, workspace='default'>]

In [8]:
client.create_experiment(name='my-very-cool-experiment')

'5'

In [4]:
runs = client.search_runs(
    experiment_ids='3',
    filter_string='metrics.rmse < 7.3',
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=5,
    order_by=['metrics.rmse ASC']
)
runs

[<Run: data=<RunData: metrics={'rmse': 7.204659011083571}, params={'learning_rate': '0.09110539397293055',
  'max_depth': '4',
  'min_child_weight': '0.6640354303569038',
  'objective': 'reg:linear',
  'reg_alpha': '0.17754186286946907',
  'reg_lambda': '0.005744546063131262',
  'seed': '42'}, tags={'mlflow.runName': 'colorful-stork-784',
  'mlflow.source.name': 'duration-prediction.ipynb',
  'mlflow.source.type': 'NOTEBOOK',
  'mlflow.user': 'danielwohlgemuth',
  'model': 'xgboost'}>, info=<RunInfo: artifact_uri='mlflow-artifacts:/3/d7a38207b10d4141b438009202de0a14/artifacts', end_time=1774831070672, experiment_id='3', lifecycle_stage='active', run_id='d7a38207b10d4141b438009202de0a14', run_name='colorful-stork-784', start_time=1774831070156, status='FINISHED', user_id='danielwohlgemuth'>, inputs=<RunInputs: dataset_inputs=[], model_inputs=[]>, outputs=<RunOutputs: model_outputs=[]>>,
 <Run: data=<RunData: metrics={'rmse': 7.235446393040263}, params={'learning_rate': '0.03661061223133

In [9]:
for run in runs:
    print(f'run id: {run.info.run_id}, rmse: {run.data.metrics["rmse"]:.4f}')

run id: d7a38207b10d4141b438009202de0a14, rmse: 7.2047
run id: 4672c7f9fe8949099a24daf270ef7253, rmse: 7.2354
run id: 769e8eadcad54359a1e1b46eb67f1b4d, rmse: 7.2354
run id: e8fd214319014ba2bbd7cb340042aa12, rmse: 7.2354
run id: c9ce571e80884212bcc5bfe7c1639e06, rmse: 7.2354


In [8]:
models = client.search_logged_models(
    experiment_ids=['3'],
    filter_string="metrics.rmse < 7.3",
    order_by=[
        {
            "field_name": "metrics.rmse",
            "ascending": False
        }
    ],
    max_results=5
)
models

[LoggedModel(artifact_location='mlflow-artifacts:/3/models/m-a1c1877670074bf2854132b81e3f05ac/artifacts', creation_timestamp=1775241890557, experiment_id='3', last_updated_timestamp=1775241895395, model_id='m-a1c1877670074bf2854132b81e3f05ac', model_type='', model_uri='models:/m-a1c1877670074bf2854132b81e3f05ac', name='models_mlflow', source_run_id='4672c7f9fe8949099a24daf270ef7253', status=<LoggedModelStatus.READY: 'READY'>, status_message=''),
 LoggedModel(artifact_location='mlflow-artifacts:/3/models/m-c7a40041f75c4c8e920a414f1b52dfd3/artifacts', creation_timestamp=1775154992906, experiment_id='3', last_updated_timestamp=1775154996510, model_id='m-c7a40041f75c4c8e920a414f1b52dfd3', model_type='', model_uri='models:/m-c7a40041f75c4c8e920a414f1b52dfd3', name='models_mlflow', source_run_id='c9ce571e80884212bcc5bfe7c1639e06', status=<LoggedModelStatus.READY: 'READY'>, status_message=''),
 LoggedModel(artifact_location='mlflow-artifacts:/3/models/m-9122bda945d24f87b1e50a4ad6755459/artifa

In [11]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

first_model = models[0]

mlflow.register_model(model_uri=first_model.artifact_location, name='nyc-taxi-regressor')

Registered model 'nyc-taxi-regressor' already exists. Creating a new version of this model...
2026/04/03 18:02:02 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: nyc-taxi-regressor, version 4
Created version '4' of model 'nyc-taxi-regressor'.


<ModelVersion: aliases=[], creation_timestamp=1775250122412, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1775250122412, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='', run_link='', source='mlflow-artifacts:/3/models/m-a1c1877670074bf2854132b81e3f05ac/artifacts', status='READY', status_message=None, tags={}, user_id='', version='4', workspace='default'>

In [16]:
models = client.search_registered_models(filter_string='name="nyc-taxi-regressor"')
models

[<RegisteredModel: aliases={'staging': '3'}, creation_timestamp=1775244125847, deployment_job_id='', deployment_job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', description='', last_updated_timestamp=1775250122412, latest_versions=[<ModelVersion: aliases=[], creation_timestamp=1775250122412, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1775250122412, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='', run_link='', source='mlflow-artifacts:/3/models/m-a1c1877670074bf2854132b81e3f05ac/artifacts', status='READY', status_message=None, tags={}, user_id='', version='4', workspace='default'>], name='nyc-taxi-regressor', tags={}, workspace='default'>]

In [28]:
model_version = client.get_model_version_by_alias(name='nyc-taxi-regressor', alias='staging')
model_version

<ModelVersion: aliases=['staging'], creation_timestamp=1775244519106, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1775244519106, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='4672c7f9fe8949099a24daf270ef7253', run_link='', source='models:/nyc-taxi-regressor/1', status='READY', status_message=None, tags={'model': 'xgboost'}, user_id='', version='3', workspace='default'>

In [30]:
client.set_registered_model_alias(name='nyc-taxi-regressor', alias='staging', version='4')

In [32]:
from datetime import datetime

date = datetime.today().date()

client.update_model_version(name='nyc-taxi-regressor', version='4', description=f'Promoted to staging on {date}')

<ModelVersion: aliases=['staging'], creation_timestamp=1775250122412, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='Promoted to staging on 2026-04-03', last_updated_timestamp=1775251724540, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='', run_link='', source='mlflow-artifacts:/3/models/m-a1c1877670074bf2854132b81e3f05ac/artifacts', status='READY', status_message=None, tags={}, user_id='', version='4', workspace='default'>

In [35]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-03.parquet
!mv green_tripdata_2021-03.parquet data/

--2026-04-04 08:34:45--  https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-03.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:2951:f000:b:20a5:b140:21, 2600:9000:2951:8600:b:20a5:b140:21, 2600:9000:2951:c200:b:20a5:b140:21, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:2951:f000:b:20a5:b140:21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1474538 (1.4M) [binary/octet-stream]
Saving to: ‘green_tripdata_2021-03.parquet’

green_tripdata_2021 100%[===================>]   1.41M  2.50MB/s    in 0.6s    

2026-04-04 08:34:47 (2.50 MB/s) - ‘green_tripdata_2021-03.parquet’ saved [1474538/1474538]

